In [0]:
storage_account = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-name")
storage_key     = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-key")
evh_conn_str    = dbutils.secrets.get(scope="fraud-kv-scope", key="evh-connection-string")
evh_namespace   = "evhb-fraud-pipeline"


In [0]:
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/transactions'
silver_checkpoint = f'abfss://silver@{storage_account}.dfs.core.windows.net/_checkpoints/transactions'
gold_path = f'abfss://gold@{storage_account}.dfs.core.windows.net/transactions'
gold_checkpoint = f'abfss://gold@{storage_account}.dfs.core.windows.net/_checkpoints/transactions'
fraud_alerts_path = f'abfss://gold@{storage_account}.dfs.core.windows.net/fraud_alerts'
fraud_alerts_checkpoint = f'abfss://gold@{storage_account}.dfs.core.windows.net/_checkpoints/fraud_alerts'
model_base_path  = f"abfss://models@{storage_account}.dfs.core.windows.net/fraud_detection"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)
THRESHOLD = 0.5 

In [0]:
import os

# Create directory if not exists
os.makedirs("/dbfs/FileStore/models", exist_ok=True)

# Save model as JSON
model.save_model("/dbfs/FileStore/models/fraud_model.json")
print("✅ Model saved to /dbfs/FileStore/models/fraud_model.json")

# Verify
print("File exists:", os.path.exists("/dbfs/FileStore/models/fraud_model.json"))
print("File size:", os.path.getsize("/dbfs/FileStore/models/fraud_model.json"), "bytes")

---------------------------------------------------------------------------
OSError                                   Traceback (most recent call last)
File <command-6661979323724602>, line 4
      1 import os
      3 # Create directory if not exists
----> 4 os.makedirs("/dbfs/FileStore/models", exist_ok=True)
      6 # Save model as JSON
      7 model.save_model("/dbfs/FileStore/models/fraud_model.json")

File /usr/lib/python3.10/os.py:215, in makedirs(name, mode, exist_ok)
    213 if head and tail and not path.exists(head):
    214     try:
--> 215         makedirs(head, exist_ok=exist_ok)
    216     except FileExistsError:
    217         # Defeats race condition when another thread created the path
    218         pass

File /usr/lib/python3.10/os.py:225, in makedirs(name, mode, exist_ok)
    223         return
    224 try:
--> 225     mkdir(name, mode)
    226 except OSError:
    227     # Cannot rely on checking for EEXIST, since the operating system
    228     # could give pri

In [0]:
import pickle
import json

dbutils.fs.cp(f'{model_base_path}/fraud_model.pkl', 'file:/tmp/fraud_model.pkl')
dbutils.fs.cp(f'{model_base_path}/features.pkl', 'file:/tmp/features.pkl')

with open('/tmp/fraud_model.pkl', 'rb') as f:
   
    print(model)
    model.save_model("/tmp/fraud_model.json")
   

with open('/tmp/features.pkl', 'rb') as f:
    features = pickle.load(f)

model_broadcast = sc.broadcast(model)
features_broadcast = sc.broadcast(features)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=200, n_jobs=None, num_parallel_tree=None,
              predictor=None, random_state=42, ...)
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='aucpr', feature_

In [0]:
@pandas_udf(DoubleType())
def predict_fraud_udf(
    amount_log            : pd.Series,
    amount_vs_avg_ratio   : pd.Series,
    is_high_amount        : pd.Series,
    amount_zscore         : pd.Series,
    hour_of_day           : pd.Series,
    is_night              : pd.Series,
    is_foreign_int        : pd.Series,
    is_suspicious_merchant: pd.Series,
    is_new_account        : pd.Series,
    txn_type_encoded      : pd.Series,
    account_age_days      : pd.Series
) -> pd.Series:
    try:
        import pickle
        import xgboost as xgb

        # Load model directly — works fine on single node
        model = xgb.XGBClassifier()
        model.load_model("/tmp/fraud_model.json'")

        features = [
            "amount_log", "amount_vs_avg_ratio", "is_high_amount",
            "amount_zscore", "hour_of_day", "is_night", "is_foreign_int",
            "is_suspicious_merchant", "is_new_account", "txn_type_encoded",
            "account_age_days"
        ]

        df = pd.DataFrame({
            "amount_log"            : amount_log,
            "amount_vs_avg_ratio"   : amount_vs_avg_ratio,
            "is_high_amount"        : is_high_amount,
            "amount_zscore"         : amount_zscore,
            "hour_of_day"           : hour_of_day,
            "is_night"              : is_night,
            "is_foreign_int"        : is_foreign_int,
            "is_suspicious_merchant": is_suspicious_merchant,
            "is_new_account"        : is_new_account,
            "txn_type_encoded"      : txn_type_encoded,
            "account_age_days"      : account_age_days
        })[features]

        probs = model.predict_proba(df)[:, 1]
        return pd.Series(probs.round(4))

    except Exception as e:
        import traceback
        with open("/tmp/udf_error.txt", "w") as f:
            f.write(traceback.format_exc())
        return pd.Series([-1.0] * len(amount_log))

In [0]:
FEATURES = [
    "amount_log", "amount_vs_avg_ratio", "is_high_amount",
    "amount_zscore", "hour_of_day", "is_night", "is_foreign_int",
    "is_suspicious_merchant", "is_new_account", "txn_type_encoded",
    "account_age_days"
]

silver_stream = (
    spark.readStream.format('delta').load(silver_path)
)
model = model_broadcast.value
features = features_broadcast.value

scored = silver_stream.withColumn('fraud_score', predict_fraud(*[col(f) for f in FEATURES])) \
    .withColumn('is_fraud', when(col('fraud_score') >= THRESHOLD, 1 ).otherwise(0)) \
    .withColumn('processed_at', current_timestamp())




In [0]:
gold_query =(
    scored.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', gold_checkpoint)
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .start(gold_path)
)


In [0]:
fraud_query = (
    scored
    .filter(col("is_fraud") == 1)        # only fraud transactions
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", fraud_alerts_checkpoint)
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .start(fraud_alerts_path)
)

In [0]:
# Check Gold table
spark.sql(f"""
    SELECT 
        transaction_id, customer_id, amount,
        fraud_score, is_fraud, processed_at
    FROM delta.`{gold_path}`
    ORDER BY fraud_score DESC
    LIMIT 10
""").show(truncate=False)

# Check Fraud alerts table
# spark.sql(f"""
#     SELECT 
#         transaction_id, customer_id, amount,
#         merchant, location, fraud_score, processed_at
#     FROM delta.`{fraud_alerts_path}`
#     LIMIT 10
# """).show(truncate=False)

+------------------------------------+-----------+--------+-----------+--------+-----------------------+
|transaction_id                      |customer_id|amount  |fraud_score|is_fraud|processed_at           |
+------------------------------------+-----------+--------+-----------+--------+-----------------------+
|c077a082-5dce-47fa-b50f-6258b1839d98|CUST0210   |2552.12 |-1.0       |0       |2026-04-05 19:47:12.123|
|6ac669f4-cda0-4fef-a52d-9b12615fc79f|CUST0371   |6564.1  |-1.0       |0       |2026-04-05 19:47:12.123|
|4408a76f-22fc-44b9-8955-21bb918777e8|CUST0339   |5224.65 |-1.0       |0       |2026-04-05 19:47:12.123|
|001657e2-c79b-4fd0-9f2f-29aa9cc75250|CUST0352   |2610.17 |-1.0       |0       |2026-04-05 19:47:12.123|
|46b195bf-8eba-4c95-834b-288648a68754|CUST0214   |17503.22|-1.0       |0       |2026-04-05 19:47:12.123|
|17e6ae1e-d816-4d36-a427-86931a614922|CUST0235   |3479.44 |-1.0       |0       |2026-04-05 19:47:12.123|
|e614f153-a997-4f45-9d7b-3297a5096d97|CUST0373   |13909

In [0]:
import pickle
import pandas as pd

# Load model locally
dbutils.fs.cp(f"{model_base_path}/fraud_model.pkl", "file:/tmp/fraud_model_test.pkl")

with open("/tmp/fraud_model_test.pkl", "rb") as f:
    model = pickle.load(f)

# Create a test row with exact feature order
test_row = pd.DataFrame([{
    "amount_log"            : 7.86,
    "amount_vs_avg_ratio"   : 2.79,
    "is_high_amount"        : 0,
    "amount_zscore"         : 1.92,
    "hour_of_day"           : 7,
    "is_night"              : 0,
    "is_foreign_int"        : 0,
    "is_suspicious_merchant": 0,
    "is_new_account"        : 0,
    "txn_type_encoded"      : 4,
    "account_age_days"      : 2085
}])

prob = model.predict_proba(test_row)[0][1]
print(f"✅ Fraud probability: {prob}")

✅ Fraud probability: 0.5300044417381287


In [0]:
silver_batch = spark.read.format("delta").load(silver_path)
result = silver_batch.withColumn(
    "fraud_score",
    predict_fraud_udf(*[col(f) for f in FEATURES])
)
result.select("transaction_id", "amount", "fraud_score").show(5)

+--------------------+--------+-----------+
|      transaction_id|  amount|fraud_score|
+--------------------+--------+-----------+
|001657e2-c79b-4fd...| 2610.17|       -1.0|
|17e6ae1e-d816-4d3...| 3479.44|       -1.0|
|4408a76f-22fc-44b...| 5224.65|       -1.0|
|e614f153-a997-4f4...|13909.24|       -1.0|
|6ac669f4-cda0-4fe...|  6564.1|       -1.0|
+--------------------+--------+-----------+
only showing top 5 rows



In [0]:
import pickle
import xgboost as xgb


model = xgb.XGBClassifier()
model.load_model("/tmp/fraud_model.json")

print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='aucpr',
              feature_types=['float', 'float', 'int', 'float', 'int', 'int',
                             'int', 'int', 'int', 'int', 'int'],
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=200, n_jobs=None, num_parallel_tree=None,
              predictor=None, random_state=42, ...)


In [0]:
with open("/tmp/udf_error.txt", "r") as f:
    print(f.read())

Traceback (most recent call last):
  File "/root/.ipykernel/1240/command-5753819973990430-3131858631", line 21, in predict_fraud_udf
  File "/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/sklearn.py", line 777, in load_model
    self.get_booster().load_model(fname)
  File "/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/core.py", line 2444, in load_model
    _check_call(_LIB.XGBoosterLoadModel(
  File "/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/core.py", line 279, in _check_call
    raise XGBoostError(py_str(_LIB.XGBGetLastError()))
xgboost.core.XGBoostError: [19:47:29] ../dmlc-core/src/io/local_filesys.cc:209: Check failed: allow_null:  LocalFileSystem::Open "/tmp/fraud_model.json'": No such file or directory
Stack trace:
  [bt] (0) /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x838bb3) [0x7f